### Ingest circuits.csv file


Step 1 - Read the CSV file using the spark dataframe reader

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
bronze_path = "abfss://f1-data@prstrgacc.dfs.core.windows.net/bronze"

circuits_df = spark.read \
.option("header", True) \
.option("inferSchema", True) \
.csv(f"{bronze_path}/circuits.csv")

display(circuits_df)

circuitId,circuitRef,name,location,country,lat,lng,alt,url
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,http://en.wikipedia.org/wiki/Sepang_International_Circuit
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,http://en.wikipedia.org/wiki/Bahrain_International_Circuit
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,http://en.wikipedia.org/wiki/Circuit_de_Barcelona-Catalunya
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,http://en.wikipedia.org/wiki/Istanbul_Park
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,http://en.wikipedia.org/wiki/Circuit_de_Monaco
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,http://en.wikipedia.org/wiki/Circuit_Gilles_Villeneuve
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,http://en.wikipedia.org/wiki/Circuit_de_Nevers_Magny-Cours
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,http://en.wikipedia.org/wiki/Silverstone_Circuit
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103,http://en.wikipedia.org/wiki/Hockenheimring


In [0]:
circuits_df.printSchema()

root
 |-- circuitId: integer (nullable = true)
 |-- circuitRef: string (nullable = true)
 |-- name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- country: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- alt: integer (nullable = true)
 |-- url: string (nullable = true)



Step 2 - Select only the required columns

In [0]:
circuits_selected_df = circuits_df.select(col("circuitId"), col("circuitRef"), col("name"), col("location"), col("country"), col("lat"), col("lng"), col("alt"))


Step 3 - Rename the columns as required

In [0]:
circuits_renamed_df = circuits_selected_df.withColumnRenamed("circuitId", "circuit_id") \
.withColumnRenamed("circuitRef", "circuit_ref") \
.withColumnRenamed("lat", "latitude") \
.withColumnRenamed("lng", "longitude") \
.withColumnRenamed("alt", "altitude") \
.withColumn("data_source", lit("kaggle"))

display(circuits_renamed_df)


circuit_id,circuit_ref,name,location,country,latitude,longitude,altitude,data_source
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,kaggle
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,kaggle
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,kaggle
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,kaggle
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,kaggle
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,kaggle
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,kaggle
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,kaggle
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,kaggle
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103,kaggle


Step 4 - Add ingestion date to the dataframe

In [0]:
circuits_final_df = circuits_renamed_df.withColumn("ingestion_date", current_timestamp())

display(circuits_final_df)


circuit_id,circuit_ref,name,location,country,latitude,longitude,altitude,data_source,ingestion_date
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,kaggle,2026-03-13T17:13:06.617394Z
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,kaggle,2026-03-13T17:13:06.617394Z
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,kaggle,2026-03-13T17:13:06.617394Z
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,kaggle,2026-03-13T17:13:06.617394Z
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,kaggle,2026-03-13T17:13:06.617394Z
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,kaggle,2026-03-13T17:13:06.617394Z
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,kaggle,2026-03-13T17:13:06.617394Z
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,kaggle,2026-03-13T17:13:06.617394Z
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,kaggle,2026-03-13T17:13:06.617394Z
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103,kaggle,2026-03-13T17:13:06.617394Z


In [0]:
%sql
-- CREATE CATALOG f1_catalog;

In [0]:
%sql
-- CREATE SCHEMA f1_catalog.f1_bronze;

-- CREATE SCHEMA f1_catalog.f1_silver;

-- CREATE SCHEMA f1_catalog.f1_gold;

In [0]:
%sql
USE CATALOG f1_catalog;

In [0]:
%sql
USE SCHEMA f1_silver;

In [0]:
circuits_final_df.write.mode("overwrite").format("delta").saveAsTable('circuits_1')


In [0]:
%sql
SHOW TABLES IN f1_catalog.f1_silver;

database,tableName,isTemporary
f1_silver,circuits_1,false


In [0]:
%sql
SELECT * FROM circuits_1

circuit_id,circuit_ref,name,location,country,latitude,longitude,altitude,data_source,ingestion_date
1,albert_park,Albert Park Grand Prix Circuit,Melbourne,Australia,-37.8497,144.968,10,kaggle,2026-03-13T17:31:24.392451Z
2,sepang,Sepang International Circuit,Kuala Lumpur,Malaysia,2.76083,101.738,18,kaggle,2026-03-13T17:31:24.392451Z
3,bahrain,Bahrain International Circuit,Sakhir,Bahrain,26.0325,50.5106,7,kaggle,2026-03-13T17:31:24.392451Z
4,catalunya,Circuit de Barcelona-Catalunya,Montmeló,Spain,41.57,2.26111,109,kaggle,2026-03-13T17:31:24.392451Z
5,istanbul,Istanbul Park,Istanbul,Turkey,40.9517,29.405,130,kaggle,2026-03-13T17:31:24.392451Z
6,monaco,Circuit de Monaco,Monte-Carlo,Monaco,43.7347,7.42056,7,kaggle,2026-03-13T17:31:24.392451Z
7,villeneuve,Circuit Gilles Villeneuve,Montreal,Canada,45.5,-73.5228,13,kaggle,2026-03-13T17:31:24.392451Z
8,magny_cours,Circuit de Nevers Magny-Cours,Magny Cours,France,46.8642,3.16361,228,kaggle,2026-03-13T17:31:24.392451Z
9,silverstone,Silverstone Circuit,Silverstone,UK,52.0786,-1.01694,153,kaggle,2026-03-13T17:31:24.392451Z
10,hockenheimring,Hockenheimring,Hockenheim,Germany,49.3278,8.56583,103,kaggle,2026-03-13T17:31:24.392451Z


In [0]:
circuits_final_df.write \
.mode("overwrite") \
.format("delta") \
.save("abfss://f1-data@prstrgacc.dfs.core.windows.net/silver/circuits")